# Azeglio Francesco Vicini's Project for Foundations of Computer Science 2026

In [1]:
import pandas as pd
import numpy as np

dropout_risk = pd.read_csv('student_dropout_risk_2026.csv')
dropout_risk.head()

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,4,16.3,0,3,8,0,6,37,5.7,0
1,2,18,Other,USA,1,1,High,1,1,0,...,3,8.4,1,1,7,0,6,15,5.4,1
2,3,24,Other,USA,0,0,High,0,1,1,...,3,11.3,1,0,2,0,3,4,4.3,0
3,4,25,Female,USA,0,1,Middle,1,1,0,...,2,13.2,1,1,5,0,7,10,6.7,0
4,5,15,Female,Canada,0,0,Low,1,1,0,...,4,7.4,1,0,7,0,10,24,5.8,0


## 1. Extract all students that study Online, have an attendance_rate smaller than 50, and have more than 5 missed_deadlines. Store them in a dataframe called at_risk.

In [2]:
at_risk = dropout_risk[(dropout_risk['study_mode'] == 'Online') & (dropout_risk['attendance_rate'] < 50) & (dropout_risk['missed_deadlines'] > 5)]
display(at_risk)

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
478,479,22,Male,Germany,0,1,High,0,0,1,...,0,13.3,1,8,8,1,3,30,6.1,1
581,582,17,Female,India,1,0,Low,1,1,1,...,3,12.4,1,11,6,0,8,0,7.1,1
642,643,29,Male,Mexico,0,0,Low,1,1,1,...,4,14.3,0,9,10,0,7,9,4.0,1
1216,1217,15,Male,UK,0,0,Low,1,1,1,...,2,6.6,0,7,7,0,5,11,5.4,1
1316,1317,21,Male,Brazil,0,0,Low,0,1,1,...,3,7.8,0,17,5,0,4,6,8.1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96590,96591,23,Female,USA,1,0,Low,1,1,1,...,4,9.6,0,10,8,0,8,0,7.9,0
96648,96649,26,Female,USA,1,0,Low,1,1,1,...,2,12.4,0,8,9,1,6,10,6.5,1
97775,97776,19,Male,Germany,0,0,Low,1,0,1,...,6,7.5,1,8,6,1,10,19,8.8,1
99696,99697,26,Female,USA,1,0,Middle,1,0,1,...,2,14.3,1,16,9,1,8,12,7.8,1


## 2. Then compute, for each course_type, how many of those students there are and what fraction of them has dropped out. Store the result in a dataframe called at_risk_by_course, with columns students and dropout_rate, sorted by decreasing dropout_rate.

In [3]:
at_risk_by_course_student = at_risk.groupby('course_type',as_index=False)['person_id'].count()
at_risk_by_course_dropout = at_risk.groupby('course_type',as_index=False)['dropout'].mean()
at_risk_by_course = pd.merge(at_risk_by_course_student,at_risk_by_course_dropout,on='course_type')
at_risk_by_course.columns = ['course_type','students','dropout_rate']
at_risk_by_course = at_risk_by_course.sort_values('dropout_rate',ascending=False)
display(at_risk_by_course)

,course_type,students,dropout_rate
3,Law & Social Sciences,29,0.896552
4,STEM,47,0.765957
1,Business,47,0.765957
0,Arts & Humanities,40,0.750000
2,Health Sciences,34,0.647059


## 3. Add to the dataframe the following columns:
- ## pass_rate: the ratio between units_passed and units_enrolled;
- ## gpa_change: the difference between gpa_semester_2 and gpa_semester_1;
- ## gpa_trend: a string that is "improving" if gpa_change is larger than 0.5, "declining" if gpa_change is smaller than -0.5, and "stable" otherwise.

In [4]:
dropout_risk['pass_rate'] = dropout_risk['units_passed'] / dropout_risk['units_enrolled']

dropout_risk['gpa_change'] = dropout_risk['gpa_semester_2'] - dropout_risk['gpa_semester_1']

condlist = [dropout_risk['gpa_change'] > 0.5,dropout_risk['gpa_change'] < -0.5]
choicelist = ['improving','declining']
dropout_risk['gpa_trend'] = np.select(condlist,choicelist,'stable')

display(dropout_risk.head())

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,3,8,0,6,37,5.7,0,1.000000,-0.89,declining
1,2,18,Other,USA,1,1,High,1,1,0,...,1,7,0,6,15,5.4,1,0.785714,-0.03,stable
2,3,24,Other,USA,0,0,High,0,1,1,...,0,2,0,3,4,4.3,0,0.823529,-0.70,declining
3,4,25,Female,USA,0,1,Middle,1,1,0,...,1,5,0,7,10,6.7,0,0.846154,-0.41,stable
4,5,15,Female,Canada,0,0,Low,1,1,0,...,0,7,0,10,24,5.8,0,0.875000,-1.05,declining


## 4. Verify that, for every student, units_passed + units_failed equals units_enrolled; if not, report the rows violating this condition.

In [5]:
violating_rows = dropout_risk[dropout_risk['units_passed'] + dropout_risk['units_failed'] != dropout_risk['units_enrolled']]
if violating_rows.empty:
    print('For every student, the sum of units_passed and units_failed equals units_enrolled.')
else:
    display(violating_rows)

For every student, the sum of units_passed and units_failed equals units_enrolled.


## 5. Compute the dropout rate for each value of gpa_trend, and the person_id of the 10 students with the largest GPA drop.

In [6]:
gpa_dropout_rate = dropout_risk.groupby('gpa_trend',as_index=False)['dropout'].mean()
gpa_dropout_rate.columns = ['gpa_trend','dropout_rate']
display(gpa_dropout_rate)

gpa_drop = dropout_risk[dropout_risk['gpa_trend'] == 'declining'].sort_values('gpa_change')
display(gpa_drop[['person_id']].head(10))

,gpa_trend,dropout_rate
0,declining,0.265086
1,improving,0.589158
2,stable,0.391231


,person_id
23225,23226
96224,96225
88489,88490
51824,51825
58935,58936
46714,46715
43035,43036
72663,72664
76705,76706
71899,71900


## 6. Partition the students into the following groups, according to weekly_work_hours: 0, 1-10, 11-20, 21-30, more than 30 (all intervals include both extremes).

In [7]:
work_hours_0 = dropout_risk[dropout_risk['weekly_work_hours'] == 0]
display(work_hours_0)
work_hours_10 = dropout_risk[dropout_risk['weekly_work_hours'].between(1,10)]
display(work_hours_10)
work_hours_20 = dropout_risk[dropout_risk['weekly_work_hours'].between(11,20)]
display(work_hours_20)
work_hours_30 = dropout_risk[dropout_risk['weekly_work_hours'].between(21,30)]
display(work_hours_30)
more_work_hours = dropout_risk[dropout_risk['weekly_work_hours'] > 30]
display(more_work_hours)

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend
1,2,18,Other,USA,1,1,High,1,1,0,...,1,7,0,6,15,5.4,1,0.785714,-0.03,stable
3,4,25,Female,USA,0,1,Middle,1,1,0,...,1,5,0,7,10,6.7,0,0.846154,-0.41,stable
4,5,15,Female,Canada,0,0,Low,1,1,0,...,0,7,0,10,24,5.8,0,0.875000,-1.05,declining
5,6,17,Female,Indonesia,1,1,Middle,1,1,0,...,2,4,1,10,8,7.5,0,0.750000,-0.55,declining
6,7,22,Female,USA,1,0,High,0,0,0,...,2,5,1,5,25,6.0,0,0.812500,-0.25,stable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,99992,15,Female,China,1,0,High,0,1,0,...,3,8,0,7,0,6.5,1,0.588235,0.00,stable
99993,99994,20,Male,USA,0,0,High,1,1,0,...,0,6,0,5,64,7.0,0,1.000000,-1.24,declining
99994,99995,19,Female,Mexico,0,1,Low,1,0,0,...,4,8,0,7,25,5.4,1,0.666667,-0.80,declining
99998,99999,21,Male,UK,0,0,High,0,1,0,...,2,4,1,10,43,6.2,0,0.714286,-0.99,declining


,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend
17,18,18,Male,China,0,1,Middle,1,1,1,...,0,7,0,5,0,5.8,0,1.000000,-1.24,declining
24,25,20,Male,USA,1,0,Low,0,0,1,...,3,6,0,6,3,6.2,0,0.833333,-0.40,stable
61,62,20,Male,Canada,0,0,Middle,1,0,1,...,1,8,0,8,34,7.7,1,0.750000,-0.76,declining
64,65,24,Male,USA,0,0,High,0,1,1,...,0,6,0,6,23,5.0,0,1.000000,-0.26,stable
78,79,24,Female,Egypt,0,0,High,1,1,1,...,5,6,1,8,8,5.6,1,0.647059,0.71,improving
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99896,99897,19,Male,UK,1,0,Low,1,0,1,...,4,8,0,4,5,6.2,0,0.687500,-0.75,declining
99932,99933,24,Male,Indonesia,0,1,Middle,0,1,1,...,0,9,0,4,5,5.0,0,0.714286,-0.14,stable
99947,99948,23,Male,Egypt,0,0,Low,1,1,1,...,9,5,0,9,4,4.5,0,0.769231,0.52,improving
99971,99972,24,Female,China,1,0,Low,1,1,1,...,5,6,0,6,46,4.7,1,0.823529,-0.99,declining


,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend
2,3,24,Other,USA,0,0,High,0,1,1,...,0,2,0,3,4,4.3,0,0.823529,-0.70,declining
12,13,22,Male,USA,0,0,Middle,0,1,1,...,3,1,0,5,10,6.8,0,0.750000,-0.98,declining
13,14,25,Female,India,1,1,Low,1,0,1,...,2,7,1,6,19,6.1,1,0.769231,-0.91,declining
18,19,25,Female,Brazil,0,1,Low,1,0,1,...,7,9,1,6,21,6.7,1,0.461538,-0.55,declining
21,22,19,Male,USA,0,0,Low,1,1,1,...,4,4,0,6,4,5.5,0,0.600000,-0.29,stable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99988,99989,24,Female,Mexico,0,0,High,1,1,1,...,3,4,0,6,54,8.2,0,1.000000,-1.15,declining
99989,99990,21,Male,USA,1,1,Low,1,1,1,...,4,10,1,4,15,6.1,1,0.562500,-0.48,stable
99992,99993,16,Female,India,0,0,High,0,1,1,...,6,4,0,5,27,5.6,0,1.000000,-0.85,declining
99995,99996,25,Female,USA,0,0,Middle,1,1,1,...,0,4,0,6,11,5.9,0,1.000000,-0.98,declining


,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,3,8,0,6,37,5.7,0,1.000000,-0.89,declining
14,15,23,Female,India,0,1,Low,1,0,1,...,4,9,1,9,10,7.1,1,0.538462,0.51,improving
29,30,23,Male,Indonesia,0,0,High,0,0,1,...,0,3,0,7,45,7.6,0,0.900000,-0.60,declining
44,45,22,Female,Nigeria,0,0,High,1,1,1,...,0,6,0,6,47,5.7,0,1.000000,-0.83,declining
45,46,22,Female,China,0,0,Low,1,1,1,...,1,10,1,4,17,4.9,0,0.900000,-0.22,stable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99968,99969,20,Female,USA,1,0,High,0,1,1,...,1,2,0,9,18,5.8,1,0.846154,-1.29,declining
99969,99970,18,Male,China,0,0,Middle,1,1,1,...,2,6,1,4,0,6.4,1,0.750000,0.03,stable
99974,99975,21,Male,USA,0,1,High,0,0,1,...,4,6,0,9,21,5.6,0,1.000000,-0.11,stable
99978,99979,22,Male,Germany,1,0,Middle,0,0,1,...,1,8,0,7,38,4.9,1,0.700000,-0.40,stable


,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend
113,114,16,Female,China,0,0,Middle,0,1,1,...,1,5,1,8,14,7.0,0,0.750000,-0.23,stable
157,158,25,Female,Egypt,0,0,High,1,1,1,...,5,10,0,7,21,6.0,1,0.700000,0.24,stable
182,183,20,Male,USA,0,0,Low,1,1,1,...,5,10,0,5,39,6.1,1,0.818182,-0.86,declining
183,184,25,Female,UK,0,0,High,1,1,1,...,7,5,0,8,33,6.5,0,0.750000,-0.12,stable
229,230,20,Male,Nigeria,0,0,High,1,1,1,...,1,6,0,10,115,4.8,0,0.941176,-1.16,declining
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99717,99718,19,Male,Philippines,1,0,Middle,1,1,1,...,4,8,0,6,64,6.3,0,0.764706,-0.28,stable
99763,99764,24,Male,China,1,1,Low,1,1,1,...,2,10,1,5,2,5.3,0,0.823529,-0.95,declining
99765,99766,24,Male,Nigeria,0,0,Low,1,1,1,...,6,7,1,7,29,8.0,1,0.600000,-0.99,declining
99839,99840,19,Male,Brazil,0,0,Middle,1,1,1,...,2,10,0,2,12,5.9,0,0.875000,-0.65,declining


## 7. Build a risk score by summing:
- ## 2 points if the tuition is not up to date;
- ## 1 point if stress_level is at least 8;
- ## 1 point if sleep_hours is smaller than 5;
- ## 1 point if the student has a part-time job with more than 20 weekly work hours;
- ## 2 points if attendance_rate is smaller than 60;
- ## 1 point if missed_deadlines is larger than 3.
## For each value of the risk score, compute the number of students and the dropout rate.

In [8]:
dropout_risk['risk_score'] = 0
dropout_risk['risk_score'] += np.select([dropout_risk['tuition_up_to_date'] == 0],[2],0)
dropout_risk['risk_score'] += np.select([dropout_risk['stress_level'] >= 8],[1],0)
dropout_risk['risk_score'] += np.select([dropout_risk['sleep_hours'] < 5],[1],0)
dropout_risk['risk_score'] += np.select([(dropout_risk['part_time_job'] == 1) & (dropout_risk['weekly_work_hours'] > 20)],[1],0)
dropout_risk['risk_score'] += np.select([dropout_risk['attendance_rate'] < 60],[2],0)
dropout_risk['risk_score'] += np.select([dropout_risk['missed_deadlines'] > 3],[1],0)

students_at_risk = dropout_risk.groupby('risk_score',as_index=False)['person_id'].count()
students_at_risk.columns = ['risk_score','students']
students_at_risk['dropout_rate'] = dropout_risk.groupby('risk_score')['dropout'].mean()

display(students_at_risk)

,risk_score,students,dropout_rate
0,0,26955,0.152847
1,1,24115,0.252084
2,2,21984,0.337200
3,3,14557,0.452703
4,4,7613,0.577433
5,5,3320,0.673494
6,6,1149,0.794604
7,7,267,0.827715
8,8,40,0.900000


## 8. Read the GDP per capita dataset published by Our World in Data directly from the URL https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true and store it in a dataframe called gdp. The column ny_gdp_pcap_pp_kd contains the GDP per capita. Clean the dataset, removing the rows that do not correspond to a single country (e.g. World, High-income countries). For each country keep only the row with the most recent year.

In [9]:
gdp = pd.read_csv('gdp-per-capita-worldbank.csv')
gdp = gdp[~(gdp['entity'].str.contains(r'World|countries|\(..\)'))]
gdp = gdp.loc[gdp.groupby('entity')['year'].idxmax()]
display(gdp)

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
24,Afghanistan,AFG,2024,1964.4523,Asia
60,Albania,ALB,2025,22724.2930,Europe
96,Algeria,DZA,2025,15882.7140,Africa
132,Andorra,AND,2025,67993.0550,Europe
168,Angola,AGO,2025,8760.1900,Africa
...,...,...,...,...,...
7264,Uzbekistan,UZB,2025,11622.8550,Asia
7300,Vanuatu,VUT,2025,3866.9980,Oceania
7336,Vietnam,VNM,2025,15457.3500,Asia
7408,Zambia,ZMB,2025,3742.2563,Africa


## 9. Join the GDP data with the students data. Compute the dropout rate for each continent.

In [10]:
dropout_risk['nationality'] = dropout_risk['nationality'].replace('UK','United Kingdom')
dropout_risk['nationality'] = dropout_risk['nationality'].replace('USA','United States')

join_df = pd.merge(gdp,dropout_risk,left_on='entity',right_on='nationality')

continent_dropout_rate = join_df.groupby('owid_region',as_index=False)['dropout'].mean()
continent_dropout_rate.columns = ['continent','dropout_rate']
display(continent_dropout_rate)

,continent,dropout_rate
0,Africa,0.316721
1,Asia,0.322908
2,Europe,0.322762
3,North America,0.317679
4,Oceania,0.321981
5,South America,0.317620
